# 06 · Summary

**The question.** Across notebooks 03 to 05, which settings made a difference on M5, and how sure can we be? This notebook collects
every verdict in one table, restates the four claims from notebook 01 with their verdicts, shows run status and caveats, and writes `FINDINGS.md`.
It reads the files that notebooks 03 to 05 write into `results/`, so run those first.

In [1]:
import sys, os
try: import m5knobs  # noqa: F401  (installed with pip install -e .)
except ImportError: sys.path.insert(0, os.path.abspath(".."))  # run from notebooks/
from m5knobs import results, data, paths, report, experiments
from m5knobs.figures import cards, stress, early_stopping, architectures
results.check_available()  # stops with instructions if results/ is missing

In [2]:
import json, pandas as pd
from IPython.display import Markdown, display
report.check_derived_fresh()  # stops unless notebooks 03-05 ran after the newest finished run
rd = lambda f: open(os.path.join(paths.RESULTS, f))
V, O, S, A = json.load(rd(report.DERIVED[0])), json.load(rd(report.DERIVED[1])), json.load(rd(report.DERIVED[2])), pd.read_csv(rd(report.DERIVED[3]))

## Every verdict

In [3]:
rows = [("setting card", c["name"], f"{c['m_label']} → {c['f_label']}", cards.label_text(V[c["key"]]["label"]))
        for c in cards.CARDS]
rows += [("clipping stress test", stress.LABELS[k].splitlines()[0], "clip off → AutoClip", stress.describe(o))
         for k, o in O.items()]
rows += [("architecture", r.model, "vs LGBM", r.read) for r in A.iloc[1:].itertuples()]
display(pd.DataFrame(rows, columns=["study", "what", "comparison", "result"]))

,study,what,comparison,result
0,setting card,Gradient clipping,"clip off → AutoClip, 10th percentile",underfit and biased
1,setting card,Input window,input_size = 13 → input_size = 52,underfit
2,setting card,Early stopping,"5 × 50 = 250 steps → 20 × 50 = 1,000 steps",underfit
3,setting card,Loss function,loss = MAE() → loss = HuberLoss(delta=5),biased
4,setting card,Input scaling,scaler_type = 'minmax' → scaler_type = 'revin',no clear effect on M5
5,clipping stress test,Learning rate 10× (1e-2),clip off → AutoClip,no measurable difference
6,clipping stress test,Learning rate 30× (3e-2),clip off → AutoClip,neither diverged; the verdict on clip off was ...
7,clipping stress test,Short-history windows let in,clip off → AutoClip,neither diverged; the verdict on clip off was ...
8,architecture,N-HiTS,vs LGBM,worse (significant)
9,architecture,TCN,vs LGBM,worse (significant)


## The claims, with their verdicts

The readings of each claim are the ones stated in notebooks 03 to 05; the text below is generated by the same functions.

In [4]:
cl, clip = cards.claims(V), V["clipping"]
clip_text = stress.conclusion(clip["label"], O)
es = pd.DataFrame([dict(window=int(lab.replace(",", "")), wape=S[a]["wape"], bias=S[a]["bias"])
                   for lab, a in early_stopping.SWEEPS[0][2]])

In [5]:
display(Markdown(f"**1. A handful of settings decide whether deep learning works.** {cl['handful']}"))
display(Markdown(f"**2. Wrong settings mostly make the model underfit.** {cl['underfit']}"))
display(Markdown(f"**3. Gradient clipping is non-negotiable for Tweedie.** {stress.headline(O)}.\n\n{clip_text}"))
display(Markdown(f"**4. Deep learning can match gradient-boosted trees.** {architectures.headline_text(A)}. "
                 f"{architectures.claim_text(A)}"))
noise = data.SEED_NOISE
display(Markdown(f"**Also, early stopping (not one of the claims).** {early_stopping.headline_text(es)}."))
display(Markdown(f"Single seed throughout; gaps under ±{noise['wape']} WAPE or ±{noise['bias']:.0f}pt bias are within "
                 "earlier seed noise."))

**1. A handful of settings decide whether deep learning works.** 4 of 5 cards gave the wrong value a verdict of underfit or biased: Gradient clipping, Input window, Early stopping and Loss function. Largest WAPE gap among them: +1.30 (Input window, 2.9% of the 'with' run's WAPE). Verdict on the claim, read as 'for most of the tested settings, the wrong value measurably hurts': holds.

**2. Wrong settings mostly make the model underfit.** 3 of 5 cards gave the wrong value a verdict of underfit: Gradient clipping, Input window and Early stopping. Verdict on the claim, read as 'more than half of the cards are underfit': holds.

**3. Gradient clipping is non-negotiable for Tweedie.** Stress test (3 setups): clipping helped in 2 and no difference in 1.

- At the recipe's settings, the card's verdict on clip off: underfit and biased.
- Learning rate 10× (1e-2): no measurable difference.
- Learning rate 30× (3e-2): neither diverged; the verdict on clip off was underfit and biased.
- Short-history windows let in: neither diverged; the verdict on clip off was underfit.
- Verdict on the claim: **holds** (recipe card: underfit and biased; in favour of clipping: clip off scored worse under the 30× learning rate and clip off scored worse with the short-history filter off).

**4. Deep learning can match gradient-boosted trees.** N-HiTS trailed LGBM by 2.67 WAPE and TCN trailed LGBM by 1.86 WAPE. No deep model was level with or better than LGBM on WAPE. Verdict on the claim: does not hold.

**Also, early stopping (not one of the claims).** More patience: 4,000 steps gave WAPE 44.92 vs 45.15 at 1,000, but bias -3.4% → -5.4%.

Single seed throughout; gaps under ±0.2 WAPE or ±1pt bias are within earlier seed noise.

## Run status

In [6]:
runs = report.run_status()
runs["device"] = [d or "not recorded" if s == "done" else "" for d, s in zip(runs.device, runs.status)]
display(Markdown(f"{(runs.status == 'done').sum()} of {len(runs)} runs finished."))
display(runs[["arm", "study", "status", "wape", "bias", "device"]].round(2))

27 of 27 runs finished.

,arm,study,status,wape,bias,device
0,f2_noclip,f2,done,49.27,8.06,mps
1,f2_clip,f2,done,48.14,6.35,mps
2,f3_fix,f3,done,45.27,-3.64,not recorded
3,f3_fix_noguard,f3,done,45.27,-3.64,not recorded
4,f3_patience250,f3,done,45.93,-0.45,not recorded
5,f3_input13,f3,done,46.57,-2.27,not recorded
6,f3_minmax,f3,done,45.12,-0.64,not recorded
7,f3_mae,f3,done,45.45,-9.83,not recorded
8,f4_lgbm,f4,done,45.35,-7.44,cpu
9,f4_nhits,f4,done,48.02,6.12,mps


## Caveats

- **One seed.** Every run uses seed 1. The seed-noise bar comes from the author's earlier 3-seed runs of the same model family
  on M5. Those runs are not part of this repo, so treat the bar as approximate.
- **Rules set after seeing results:** the effect floor in the verdict rule, and the stress test's divergence rule (notebook 01).
- **Card forecasts** at the second and third origins come from the first origin's fit; the architecture table refits at every
  origin, so its N-HiTS numbers are not directly comparable with the cards'.
- **Loss in the cards:** four of the five cards use Huber δ5, not the recipe's Tweedie, so the scaler card isn't confounded
  by the Tweedie link's dependence on RevIN's window mean (notebook 03). On MinMax's [0, 1] scale Huber δ5 never
  caps an error, so the MinMax arm effectively trains on squared error.
- **Tweedie link floor:** the stock link, which the guard keeps, keeps the forecast mean at or above about one unit a week and
  at or above the input window's mean, both in sales units (notebook 02). Notebook 05 shows forecasts on slow sellers.
- **Series with no sales before an origin** (item-store pairs whose first sale came after it) get no neural forecast and
  are scored as zero; the count is below.
- **Devices:** reruns on the same machine and device were bit-identical in the author's checks; across devices expect small
  differences, possibly larger for the high-learning-rate stress runs. Runs made by the current runner record their device;
  older runs show "not recorded" in the run-status table.


In [7]:
u = report.unforecast()
display(Markdown(f"Series with no sales before an origin: {u['per_min']:,} to {u['per_max']:,} per origin, "
                 f"{u['units'] / u['total']:.2%} of scored volume, the same for every neural run."))

Series with no sales before an origin: 58 to 82 per origin, 0.13% of scored volume, the same for every neural run.

## Write FINDINGS.md

Every number behind the figures, as a Markdown file at the repo root.

In [8]:
out = report.write()
display(Markdown(f"Wrote `{os.path.relpath(out)}`."))

Wrote `../FINDINGS.md`.

## What to set

In [9]:
lines = [f"- `{c['knob']}` ({cards.label_text(V[c['key']]['label'])}): {V[c['key']]['try_line']}" for c in cards.CARDS]
display(Markdown("The recipe in one place is `m5knobs.model.Knobs` (notebook 02). Per setting, from the cards in "
                 "notebook 03 (verdict on the wrong value, then the advice):\n\n" + "\n".join(lines) + "\n\n"
                 f"Notebook 04: {stress.headline(O)}. Notebook 05: {architectures.headline_text(A)}."))

The recipe in one place is `m5knobs.model.Knobs` (notebook 02). Per setting, from the cards in notebook 03 (verdict on the wrong value, then the advice):

- `callbacks=[AutoClip(clip_percentile)]` (underfit and biased): Turn it on for Tweedie: AutoClip at the 10th percentile, or a fixed gradient_clip_val.
- `input_size` (underfit): Start at one season (52 weekly, 12 monthly) or two, and shorten only if you have evidence.
- `early_stop_patience_steps × val_check_steps` (underfit): Allow about 1,000 optimizer steps without improvement before stopping; a longer window can trade WAPE for bias, so check both.
- `loss` (biased): Pick the loss by how you're scored: if total volume (bias) matters, avoid pure MAE; tune Huber's delta.
- `scaler_type` (no clear effect on M5): Compare RevIN, robust and MinMax on your own data before settling on one.

Notebook 04: Stress test (3 setups): clipping helped in 2 and no difference in 1. Notebook 05: N-HiTS trailed LGBM by 2.67 WAPE and TCN trailed LGBM by 1.86 WAPE.

Back to the start: [01 · The problem](01_the_problem.ipynb).